# FinSafe AI: Exploratory Financial Distress Analysis & Anomaly Modeling
This notebook demonstrates the unsupervised stress anomaly detection pipeline for early borrower distress identification.

### Pipeline Overview:
1. **Synthetic Portfolio Generation**: Simulating borrower cash flow, EMI commitments, deal purchases, and liquidity burn.
2. **Feature Engineering**: Deriving non-linear stress metrics (Spend-to-Income, Deal Reliance, Liquidity Runway).
3. **Unsupervised Isolation Forest**: Detecting multi-dimensional anomalies prior to formal delinquency.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

from src.data.loader import generate_synthetic_customers
from src.data.feature_engineering import compute_stress_features, get_feature_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries successfully imported!")

## 1. Portfolio Ingestion & Feature Engineering

In [ ]:
df_raw = generate_synthetic_customers(n_samples=500, random_seed=42)
df_features = compute_stress_features(df_raw)
print(f"Generated {len(df_features)} borrower profiles.")
df_features[['customer_id', 'monthly_income', 'current_emi', 'spend_to_income_ratio', 'liquidity_runway_months', 'stress_index']].head()

## 2. Exploratory Distress Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_features['spend_to_income_ratio'], kde=True, ax=axes[0], color='#6366f1')
axes[0].set_title('Spend-to-Income Ratio Distribution')
axes[0].axvline(0.70, color='red', linestyle='--', label='Stress Boundary (70%)')
axes[0].legend()

sns.histplot(df_features['liquidity_runway_months'], kde=True, ax=axes[1], color='#06b6d4')
axes[1].set_title('Liquidity Runway (Months)')
axes[1].axvline(1.5, color='orange', linestyle='--', label='Critical Buffer (1.5 Mos)')
axes[1].legend()

sns.histplot(df_features['deal_reliance_index'], kde=True, ax=axes[2], color='#f59e0b')
axes[2].set_title('Deal & Coupon Reliance Index')
axes[2].axvline(0.60, color='red', linestyle='--', label='Distress Behavior Threshold')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Unsupervised Isolation Forest Modeling

In [ ]:
X = get_feature_matrix(df_features)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(n_estimators=150, contamination=0.15, random_state=42)
model.fit(X_scaled)

df_features['anomaly_raw'] = model.decision_function(X_scaled)
df_features['is_anomaly'] = model.predict(X_scaled) == -1

print("Anomaly Detection Summary:")
print(df_features['is_anomaly'].value_counts(normalize=True))

## 4. Anomaly Decision Boundary & Clustering

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_features,
    x='spend_to_income_ratio',
    y='deal_reliance_index',
    hue='is_anomaly',
    palette={False: '#10b981', True: '#f43f5e'},
    alpha=0.8,
    s=60
)
plt.title('Early Stress Anomaly Cluster Isolation')
plt.xlabel('Monthly Spend / Income')
plt.ylabel('Emergency Deal Reliance Index')
plt.show()